# 🔧 Build a supervisor + specialists team



> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 17 §17.4–§17.5 · type: walkthrough*



**The promise:** you'll stand up the chapter's 🔧 Build — a supervisor that owns the goal and budget, a researcher wired to a RAG tool, and a writer with no retrieval — with each specialist exposed to the supervisor *as a tool*. The capstone's first team.

## 🧠 Why this matters



The use case: a user asks for a researched brief on a topic covered by the company's private documents. One agent does this passably — but cramming retrieval noise and writing guidelines into one context degrades both.



So we split it for two named forces: **context isolation** (retrieval noise stays out of the writing context) and **privilege separation** (only the researcher holds the retrieval tool). The cleanest way to reach a team in a raw-SDK world is the move you already know: **each specialist is exposed to the supervisor as a tool.** Delegation becomes a tool call; the handoff schema becomes the tool's input schema; budgets and logging live in the tool executor — all machinery from Ch 12 and Ch 16.

## Objectives & prereqs



**By the end you can:**

- Implement a `Specialist` whose `as_tool` exposes the handoff schema as its `input_schema`.

- Run a `supervise` loop — the Ch 12 tool loop where the *tools are the team*.

- See context isolation and a hierarchical `RunBudget` actually working, not just described.



**Prereqs:** `17-01` (topologies, the `Handoff` schema); Ch 12 (the tool loop the supervisor *is*); Ch 16 (`RunBudget`); Ch 13 (the `search_docs` RAG tool); Ch 15 (handoff schema as tool input).



**Cost:** runs free in `MOCK=1` (default) — the mock scripts the supervisor's delegations and the specialist outputs. `MOCK=0` would cost a few supervisor turns plus one call per specialist invocation.

In [ ]:
# Setup. MOCK=1 (default) -> no key, no network, deterministic.

import os

import json

import random

from pathlib import Path



from dotenv import load_dotenv



load_dotenv()

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

MODEL = "claude-opus-4-8"  # the book's default; only used on the MOCK=0 path



random.seed(17)  # the mock 'LLM' is scripted, but seed anything stochastic anyway



# The live path is opt-in and fails fast with a friendly message if the key is missing.

client = None

if not MOCK:

    if not os.getenv("ANTHROPIC_API_KEY"):

        raise SystemExit("MOCK=0 needs ANTHROPIC_API_KEY in your environment (.env). "

                         "Set COMPANION_MOCK=1 to run free and offline.")

    import anthropic

    client = anthropic.Anthropic()



print(f"MOCK={MOCK}  model={MODEL}  (MOCK=1 -> free & offline)")

### The document base and the RAG tool (Ch 13)



The researcher's only skill is `search_docs`, the platform's retrieval tool. We back it with a tiny committed fixture (`data/docs.json`) and a keyword match — enough to make *citations* real without a vector DB or any network.

In [ ]:
# Load the tiny doc base committed under data/.

DOCS = json.loads((Path("data") / "docs.json").read_text(encoding="utf-8"))

print(f"Loaded {len(DOCS)} docs:", ", ".join(DOCS))



# The RAG tool's wire schema, exactly as the supervisor/researcher would see it (Ch 13).

SEARCH_DOCS_TOOL = {

    "name": "search_docs",

    "description": "Search the company document base. Returns matching docs with their ids.",

    "input_schema": {

        "type": "object",

        "properties": {"query": {"type": "string"}},

        "required": ["query"],

    },

}



def run_rag_tool(name: str, args: dict) -> str:

    # Offline keyword retrieval over the fixture. Real version: vector search (Ch 13).

    q = args.get("query", "").lower()

    hits = [d for d in DOCS.values()

            if any(w in d["text"].lower() or w in d["title"].lower()

                   for w in q.split() if len(w) > 3)]

    if not hits:

        return "No matching documents."

    return "\n".join(f"[{d['id']}] {d['title']}: {d['text']}" for d in hits[:3])



print(run_rag_tool("search_docs", {"query": "retention churn analytics"})[:120], "...")

### The hierarchical budget (Ch 16)



`RunBudget` from Ch 16 becomes *hierarchical* here: the **same** budget object is threaded through the supervisor *and* every specialist, so one runaway worker can't consume the whole run. We count agent **steps** (each is a model call) rather than tokens to keep the mock deterministic; the real one charges `resp.usage`.

In [ ]:
class BudgetExceeded(Exception):

    pass



class RunBudget:

    """One shared budget for the whole tree (supervisor + specialists)."""

    def __init__(self, max_steps: int):

        self.max_steps = max_steps

        self.spent = 0

        self.ledger = []  # (actor, step) -> a readable per-role trace



    def raise_if_spent(self):

        if self.spent >= self.max_steps:

            raise BudgetExceeded(f"run budget of {self.max_steps} steps exhausted")



    def charge(self, action_sig: str, steps: int = 1):

        self.spent += steps

        self.ledger.append((action_sig, self.spent))



budget = RunBudget(max_steps=12)

print("budget ready:", budget.max_steps, "steps shared across the whole team")

## The mock 'LLM' (what makes this run free)



In `MOCK=1` we replace `client.messages.create(...)` with a scripted responder. It returns the same *shape* the Anthropic SDK returns — content blocks with `.type`, a `.stop_reason`, tool-use blocks with `.name`/`.input`/`.id` — so the supervisor and specialist loops below are the **real** code, not a toy. Swap `MOCK=0` and the identical loops hit the live API.

In [ ]:
# Minimal SDK-shaped response objects so the loops are provider-real.

class _Text:

    type = "text"

    def __init__(self, text): self.text = text



class _ToolUse:

    type = "tool_use"

    def __init__(self, name, inp, id): self.name, self.input, self.id = name, inp, id



class _Resp:

    def __init__(self, content, stop_reason): self.content, self.stop_reason = content, stop_reason



# A scripted plan: supervisor delegates to researcher, then writer, then integrates.

# Keyed by (actor, turn-index) so each role's calls are deterministic.

def mock_create(actor, messages, tools):

    turn = sum(1 for m in messages if m["role"] == "assistant")

    if actor == "supervisor":

        if turn == 0:

            return _Resp([_ToolUse("researcher", {

                "task": "Find the cause of the Q3 retention decline, with sources.",

                "context": "User wants a brief on why Q3 retention fell. You have search_docs.",

                "done_criteria": "Bullet findings, each with a source id."}, "t1")], "tool_use")

        if turn == 1:

            return _Resp([_ToolUse("writer", {

                "task": "Write a one-paragraph brief on the Q3 retention decline.",

                "context": _LAST_RESEARCH[0],  # supervisor passes the FINDINGS, not raw docs

                "done_criteria": "One paragraph; keep every [doc-xxx] citation marker."}, "t2")], "tool_use")

        return _Resp([_Text(_LAST_DRAFT[0])], "end_turn")  # integrate & finish

    if actor == "researcher":

        if turn == 0:

            return _Resp([_ToolUse("search_docs",

                                   {"query": "retention churn analytics add-on pricing"}, "r1")], "tool_use")

        findings = ("- NRR fell to 104% from 119%, driven by mid-market churn [doc-001]\n"

                    "- 61% of churned mid-market accounts cited unclear ROI on the analytics add-on [doc-001]\n"

                    "- Only 22% of add-on buyers activated >1 dashboard in 90 days; activation predicts renewal [doc-002]\n"

                    "- The March change moved the add-on to per-seat pricing, raising prices for small teams [doc-003]")

        _LAST_RESEARCH[0] = findings

        return _Resp([_Text(findings)], "end_turn")

    if actor == "writer":

        draft = ("Q3 net revenue retention fell to 104% from 119% [doc-001], driven almost "

                 "entirely by mid-market churn following the March move to per-seat add-on "

                 "pricing [doc-003]. Exit surveys point to weak ROI on the analytics add-on "

                 "(cited by 61% of churned accounts [doc-001]), consistent with telemetry "

                 "showing only 22% of buyers activated more than one dashboard in 90 days, "

                 "the strongest predictor of renewal [doc-002].")

        _LAST_DRAFT[0] = draft

        return _Resp([_Text(draft)], "end_turn")

    raise ValueError(actor)



_LAST_RESEARCH = [""]  # tiny mailboxes the script uses to thread real outputs forward

_LAST_DRAFT = [""]

print("mock LLM ready")

## 🔧 The `Specialist`: a focused agent the supervisor invokes like a tool



A `Specialist` is a focused agent. Its `as_tool` is exactly how the supervisor *sees* it — a tool whose `input_schema` is the handoff (`task`, `context`, `done_criteria`). Its `run` is the Ch 12 tool loop, with the shared budget charged on every model call.

In [ ]:
class Specialist:

    """A focused agent the supervisor can invoke like a tool (book §17.5)."""

    def __init__(self, name, description, system, tools=(), run_tool=None):

        self.name, self.system = name, system

        self.tools, self.run_tool = list(tools), run_tool

        self.as_tool = {                       # how the supervisor sees it

            "name": name,

            "description": description,

            "input_schema": {                  # the Handoff schema, as a tool input

                "type": "object",

                "properties": {

                    "task":    {"type": "string"},

                    "context": {"type": "string"},

                    "done_criteria": {"type": "string"},

                },

                "required": ["task", "context", "done_criteria"],

            },

        }



    def run(self, handoff: dict, budget) -> str:

        # The specialist sees ONLY its handoff — never the supervisor's full context.

        prompt = (f"Task: {handoff['task']}\n"

                  f"Context: {handoff['context']}\n"

                  f"Done when: {handoff['done_criteria']}")

        messages = [{"role": "user", "content": prompt}]

        while True:

            budget.raise_if_spent()            # Ch 16: hierarchical budget

            if MOCK:

                resp = mock_create(self.name, messages, self.tools)

            else:

                resp = client.messages.create(

                    model=MODEL, max_tokens=4096, system=self.system,

                    tools=self.tools, messages=messages)

            budget.charge(action_sig=self.name)

            if resp.stop_reason != "tool_use":

                return next((b.text for b in resp.content if b.type == "text"), "")

            messages.append({"role": "assistant", "content": resp.content})

            messages.append({"role": "user", "content": [

                {"type": "tool_result", "tool_use_id": b.id,

                 "content": self.run_tool(b.name, b.input)}

                for b in resp.content if b.type == "tool_use"]})



print("Specialist defined")

### Two specialists that differ only in job description and access



Same class, different *employees*. The `researcher` holds the RAG tool and must cite source ids; the `writer` has **no retrieval** (privilege separation) and must keep every citation marker.

In [ ]:
researcher = Specialist(

    name="researcher",

    description="Finds and cites facts from the company document base. "

                "Delegate all fact-finding here.",

    system="You are a research specialist. Use search_docs to gather evidence. "

           "Return findings as bullet points, each with its source document id. "

           "Never state a fact without a source.",

    tools=[SEARCH_DOCS_TOOL],          # the RAG tool from Ch 13

    run_tool=run_rag_tool,

)



writer = Specialist(

    name="writer",

    description="Turns research notes into a polished brief. Delegate all drafting "

                "here; pass the notes in 'context'.",

    system="You are a writing specialist. Write a clear, structured brief from the "

           "notes provided. Keep every citation marker. Do not invent facts beyond the notes.",

    # no tools -> no retrieval. Privilege separation, by construction.

)



team = [researcher, writer]

print("writer has retrieval?", bool(writer.tools), "| researcher has retrieval?", bool(researcher.tools))

## 🔧 The `supervise` loop: the Ch 12 tool loop, where the tools are the team



The supervisor is *just* a tool loop — but its tools are `[s.as_tool for s in team]`. A delegation is a tool call; the result integrates straight back into the conversation. The **same** `budget` is passed down into each `specialist.run(...)`, which is what makes the budget hierarchical.

In [ ]:
def supervise(goal: str, team, budget) -> str:

    by_name = {s.name: s for s in team}

    messages = [{"role": "user", "content": goal}]

    system = ("You are the supervisor. Decompose the goal, delegate to your specialists "

              "via their tools, and integrate results. Give each delegation full context: "

              "specialists share no memory with you or each other.")

    while True:

        budget.raise_if_spent()

        if MOCK:

            resp = mock_create("supervisor", messages, [s.as_tool for s in team])

        else:

            resp = client.messages.create(

                model=MODEL, max_tokens=4096, system=system,

                tools=[s.as_tool for s in team], messages=messages)

        budget.charge(action_sig="supervisor")

        if resp.stop_reason != "tool_use":

            return next((b.text for b in resp.content if b.type == "text"), "")

        messages.append({"role": "assistant", "content": resp.content})

        # Each delegation calls a teammate's run(), threading the SAME budget down the tree.

        messages.append({"role": "user", "content": [

            {"type": "tool_result", "tool_use_id": b.id,

             "content": by_name[b.name].run(b.input, budget)}

            for b in resp.content if b.type == "tool_use"]})



print("supervise loop defined")

## 🔮 Predict: run the team



We'll run `supervise("Write a brief on why Q3 retention declined...")`.



**Predict before you run the next cell:**

1. How many model **steps** will the whole run charge to the shared budget? (Count: supervisor turns + researcher turns + writer turns.)

2. Will the final brief carry `[doc-xxx]` citation markers, even though the *writer* never touched the retrieval tool?

In [ ]:
goal = ("Write a one-paragraph, fully-cited brief on why Q3 net revenue "

        "retention declined, using the company documents.")



budget = RunBudget(max_steps=12)

brief = supervise(goal, team, budget)



print("FINAL BRIEF:\n", brief)

print("\nSteps charged:", budget.spent, "of", budget.max_steps)

print("Citations present in writer's output?", "[doc-" in brief)

In [ ]:
# The budget ledger is a readable per-role trace (Ch 23): who spent what, in order.

for actor, running_total in budget.ledger:

    print(f"  step {running_total:>2}: {actor}")

**What you just saw.** The writer produced citations it could not have retrieved — because the *supervisor* passed the researcher's findings (with ids) in the handoff `context`. That is the structured handoff doing its job: the writer needs **no** retrieval tool, only the notes. Privilege separation and context isolation, both visible in one run.

## Context isolation, made visible



Each specialist sees **only** its handoff — never the supervisor's full conversation, and never each other's. Let's print exactly what the writer was handed: research *notes*, not the raw retrieved documents. Retrieval noise stays out of the writing context.

In [ ]:
# Reconstruct the writer's handoff (turn 1 of the scripted supervisor).

writer_handoff = {

    "task": "Write a one-paragraph brief on the Q3 retention decline.",

    "context": _LAST_RESEARCH[0],   # the FINDINGS the supervisor forwarded

    "done_criteria": "One paragraph; keep every [doc-xxx] citation marker.",

}

print("What the WRITER sees (its entire world):\n")

print(f"  task:    {writer_handoff['task']}")

print(f"  context: {writer_handoff['context'][:90]}...")

print("\nWhat the writer does NOT see: the raw docs, the supervisor's goal text, "

      "or the researcher's tool calls.")

# Proof it never saw raw retrieval: the full doc text isn't in its context.

print("Raw doc text leaked into writer context?",

      DOCS["doc-001"]["text"] in writer_handoff["context"])  # -> False

## ⚠️ Pitfall: an under-specified handoff (specialists share no memory)



🔮 **Predict:** if the supervisor hands the writer a *thin* context — just "write up the retention thing" — what does the writer produce? Remember: the writer has no retrieval and no memory of the research. It can only work from the handoff.

In [ ]:
# A deliberately thin handoff: the supervisor under-specifies the context.

thin_handoff = {

    "task": "Write up the retention thing.",

    "context": "Retention went down in Q3. Mid-market.",  # the numbers/sources are GONE

    "done_criteria": "A paragraph.",

}

# The writer can't cite what it was never given -> it solves 'adjacent-X', not X.

thin_result = ("Q3 retention declined, with weakness concentrated in the mid-market "

               "segment.")  # vague, uncitable, sources lost

print("Thin-handoff brief:\n ", thin_result)

print("\nHas citations?", "[doc-" in thin_result, "| Has the 104%/119% numbers?",

      "104" in thin_result)

In [ ]:
# The fix: a RICHER context. Specialists share no memory, so the handoff must carry

# everything the receiver needs -- exactly the Handoff discipline from 17-01.

rich_handoff = {

    "task": "Write a one-paragraph, fully-cited brief on the Q3 retention decline.",

    "context": _LAST_RESEARCH[0],   # the full findings, WITH source ids

    "done_criteria": "One paragraph; keep every [doc-xxx] citation marker.",

}

print("Rich context carries the numbers + sources?",

      "104%" in rich_handoff["context"] and "[doc-001]" in rich_handoff["context"])  # -> True

**What you just saw.** The worker didn't get dumber — it got *under-briefed*. The most common multi-agent failure is a lost message at the interface, and the supervisor owns the interface. Spend your effort on the handoff `context`, not on a cleverer writer prompt.

## Coordination notes (§17.4): why the supervisor topology *is* concurrency control



A multi-agent system **is** a distributed system: concurrent workers, shared mutable state, partial failure. The supervisor topology wins in production largely because it is a **concurrency-control device** — one component decomposes tasks, assigns owners, and **serializes conflicting writes** through itself. Contention collapses to the supervisor's queue discipline (a problem databases solved decades ago) instead of a free-for-all.



Two disciplines carry over wholesale:

- **Idempotent worker tasks** — a re-dispatched task must be safe to run twice (at-least-once delivery applies to agents too).

- ⚠️ **Propagate budgets down the tree** — the *same* `RunBudget` in supervisor *and* specialists, so one runaway worker can't starve the team.

In [ ]:
# Prove the budget is genuinely hierarchical: a tiny budget trips mid-delegation,

# inside a specialist's loop -- not just at the supervisor's top level.

tiny = RunBudget(max_steps=2)  # not enough for supervisor + researcher + writer

try:

    supervise(goal, team, tiny)

except BudgetExceeded as e:

    print("Stopped early, as designed:", e)

    print("Spent before stopping:", tiny.spent, "steps  | trace:", tiny.ledger)

## 🎯 Senior lens



You can't fix a team you can't see. The non-negotiable senior discipline here is **trace every hop and eval per-role** (Ch 22/23), not just end-to-end — when the brief is wrong, you need to know *which member* failed: did the researcher miss a source, or did the writer drop a citation the supervisor handed it? The budget ledger above is the seed of that per-role trace. And keep re-asking the one-agent question: this team is justified by context isolation and privilege separation — the moment those forces stop applying, collapse it back. Sixty lines of orchestration is the right amount; reach for a framework (Ch 18) only when the machinery you'd hand-roll exceeds it.

## Recap



- A `Specialist` is a focused agent; `as_tool` exposes the **handoff schema** as its `input_schema`, so delegation is just a tool call.

- `supervise` is the **Ch 12 tool loop** where the tools are the team; results integrate straight back.

- **Context isolation**: each specialist sees only its handoff — the writer cited sources it never retrieved, because the supervisor forwarded the notes.

- **Privilege separation**: only the researcher holds retrieval; the writer has no tools.

- The **same** `RunBudget`, threaded through supervisor and specialists, makes termination hierarchical — a runaway worker can't consume the run.

- The dominant failure is an **under-specified handoff**; the supervisor owns that interface.

## Exercises



1. **Add a critic specialist.** Give the team a `critic` (no retrieval) that the supervisor calls *after* the writer to check every claim has a citation. Extend the mock script so the supervisor delegates writer → critic. 🔮 Predict how many steps the run now charges.

2. **Make a worker task idempotent.** The researcher is re-dispatched (at-least-once delivery). Add a cache so a repeated identical handoff returns the prior findings without re-charging a `search_docs` call. Show the budget is lower on the second run.

3. **Tighten the budget.** Find the *smallest* `max_steps` that still lets the full researcher→writer→integrate run finish. What does that number tell you about the team's minimum cost?

4. **Break privilege separation, then restore it.** Temporarily give the `writer` the RAG tool. What invariant did you just lose, and how would per-role tracing catch a writer that starts retrieving on its own?

In [ ]:
# Exercise 1: add a 'critic' specialist; extend mock_create for writer -> critic


In [ ]:
# Exercise 2: idempotent researcher via a handoff cache; show lower budget on re-run


In [ ]:
# Exercise 3: find the minimum max_steps that still completes the run


In [ ]:
# Exercise 4: give 'writer' the RAG tool, observe the lost invariant, then revert


## Next



You built the 60-line version. Here's the real one — typed handoffs, hierarchical budgets, per-role tracing, and a task board with atomic claiming:



- 🔧 **Blueprint (production version):** [`../../blueprints/multi-agent-supervisor/`](../../blueprints/multi-agent-supervisor/)

- 🏁 **Capstone:** this is the seed of [`../../capstone-project/agents/`](../../capstone-project/agents/) — the supervisor + researcher + writer team. Checkpoint `ch17-supervisor`.

- 📐 **Template:** the `Specialist`/`Handoff` shapes feed [`../../templates/agent-project-starter/`](../../templates/agent-project-starter/).

- 📘 **Book:** §17.4–§17.5. Next chapter (Ch 18) rebuilds this same team with three frameworks; Ch 20 gates its riskier tools; Part VII runs the supervisor as a Celery job behind FastAPI.